<a href="https://colab.research.google.com/github/pachterlab/cellmender/blob/main/benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# try:
#     from cellsweep import denoise_count_matrix
# except ImportError:
#     print("cellsweep not found, installing...")
#     !pip install -U -q cellsweep[analysis]

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import itertools
import yaml
import requests
import matplotlib.pyplot as plt
import anndata as ad
from collections import OrderedDict
import seaborn as sns
import scanpy as sc
from cellsweep import denoise_count_matrix
import cellsweep.utils as cs_utils

cellsweep_dir = os.path.dirname(os.path.abspath(""))

# Compare CellBender vs. cellsweep

Some datasets of use:
- pbmc8k: 8k PBMCs from a healthy donor (CellBender Fig2): https://www.10xgenomics.com/datasets/8-k-pbm-cs-from-a-healthy-donor-2-standard-2-1-0
  - see run configuration on page 13 (bottom left) of the [Cellbender manuscript](https://doi.org/10.1038/s41592-023-01943-7)
- hgmm12k: Human-mouse mixture (CellBender Fig5): https://support.10xgenomics.com/single-cell-gene-expression/datasets/2.1.0/hgmm_12k

In [ ]:
adata_raw_parent_dir = "/mnt/data1"
adata_filtered_dir = "/mnt/data1/8_cube"

debug = False
verbose = 2  # 2 debug, 1 info, 0 warning, -1 error, -2 critical
overwrite = False  # overwrite existing files
threads = 8  # for cellsweep and CellBender (if use_cuda=False)

In [ ]:
dataset_name = "8cubed"

# create directories for data, output
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", dataset_name)
os.makedirs(data_dir, exist_ok=True)

out_dir = os.path.join(cellsweep_dir, "notebooks", "output", dataset_name)
os.makedirs(out_dir, exist_ok=True)

eight_cubed_markers_path = os.path.join(data_dir, "8_cube_marker_genes.csv")

cellsweep_max_iter = 500
cellsweep_beta = 0.1
cellsweep_init_alpha = 0.9

backed = "r" if debug else None

if not os.path.exists(adata_raw_parent_dir):
    raise ValueError(f"adata_raw_parent_dir {adata_raw_parent_dir} does not exist.")
if not os.path.exists(adata_filtered_dir):
    raise ValueError(f"adata_filtered_dir {adata_filtered_dir} does not exist.")

In [ ]:
plate_to_tissues = {}
plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]
for plate in plates:
    plate_dir = os.path.join(adata_raw_parent_dir, plate)
    plate_to_tissues[plate] = [tissue for tissue in os.listdir(plate_dir)]
        
plate_to_tissues

## Raw

In [ ]:
def load_8cubed_raw_data(adata_dir, debug=False):
    # plate: adata
    try:
        from tqdm import tqdm
        tq = tqdm
    except ImportError:
        # no tqdm → identity function
        tq = lambda x, *args, **kwargs: x

    plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]

    backed = "r" if debug else None

    adata_dict = {}
    for plate in tq(plates, desc="Loading AnnData plates"):
        adatas_plate = []
        plate_dir = os.path.join(adata_dir, plate)
        if len(os.listdir(plate_dir)) < 2 or len(os.listdir(plate_dir)) > 3:
            print(f"[WARN] Plate directory {plate_dir} does not contain 2-3 tissues, skipping")
            continue
        for tissue in os.listdir(plate_dir):
            adata_path = os.path.join(plate_dir, tissue, "adata.h5ad")
            adata = ad.read_h5ad(adata_path, backed=backed)
            if debug:
                adata = adata[:10_000, :].to_memory()
            adata.var_names_make_unique()
            # merge GonadsMale and GonadsFemale into Gonads
            adata.obs["Tissue_original"] = adata.obs["Tissue"]
            adata.obs["Tissue"] = adata.obs["Tissue"].replace({
                "GonadsMale": "Gonads",
                "GonadsFemale": "Gonads",
            })
            adatas_plate.append(adata)
        adata_plate = ad.concat(adatas_plate, axis=0, join="outer", label="Tissue", keys=[tissue for tissue in os.listdir(plate_dir)])
        adata_dict[plate] = adata_plate
    return adata_dict

def load_8cubed_filtered_data(adata_dir, debug=False):
    # tissue: adata
    try:
        from tqdm import tqdm
        tq = tqdm
    except ImportError:
        # no tqdm → identity function
        tq = lambda x, *args, **kwargs: x
    
    backed = "r" if debug else None

    # List .h5ad files
    files = sorted([os.path.join(adata_dir, f) for f in os.listdir(adata_dir) if f.endswith(".h5ad")])

    if not files:
        raise ValueError(f"No .h5ad files found in: {adata_dir}")

    print(f"Found {len(files)} files:")
    for f in files:
        print("  -", f)

    # Load them
    adata_dict = {}
    for filename in tq(files, desc="Loading AnnData files"):
        tissue = os.path.basename(filename).replace("_annotated.h5ad", "")
        adata = ad.read_h5ad(filename, backed=backed)
        if debug:
            adata = adata[:1_000, :].to_memory()
        adata.var_names_make_unique()
        # merge GonadsMale and GonadsFemale into Gonads
        adata.obs["Tissue_original"] = adata.obs["Tissue"]
        adata.obs["Tissue"] = adata.obs["Tissue"].replace({
            "GonadsMale": "Gonads",
            "GonadsFemale": "Gonads",
        })
        # ensure correct index
        if "cellID" in adata.obs.columns:
            adata.obs.set_index("cellID", inplace=True)
        adata_dict[tissue] = adata
    
    # combine gonads
    adata_dict["Gonads"] = ad.concat([adata_dict["GonadsMale"], adata_dict["GonadsFemale"]], join="outer", merge="unique", label=None)
    del adata_dict["GonadsMale"]
    del adata_dict["GonadsFemale"]

    return adata_dict

def merge_filtered_celltype_into_raw(adata_raw_dict, adata_filtered_tissue_dict):
    adata_raw_plate_with_celltype_dict = {}

    for plate, adata in adata_raw_dict.items():
        if "celltype" in adata.obs.columns:
            print(f"[WARN] Plate {plate} already has celltype column — skipping")
            continue

        # Make a copy so original isn't modified
        # adata = adata.copy()

        # Start by initializing the column with Empty Droplet everywhere
        adata.obs["celltype"] = "Empty Droplet"

        # Loop over tissues for this plate
        for tissue in adata.obs["Tissue"].unique():

            if tissue not in adata_filtered_tissue_dict:
                print(f"[WARN] Tissue {tissue} not in filtered dict — skipping")
                continue

            adata_filtered = adata_filtered_tissue_dict[tissue]

            if "celltype" not in adata_filtered.obs.columns:
                print(f"[WARN] Filtered {tissue} has no celltype column — skipping")
                continue

            # Filtered celltypes index
            filt_obs = adata_filtered.obs[["celltype"]].copy()

            # Only cells that exist in this raw plate
            common_idx = adata.obs.index.intersection(filt_obs.index)

            if len(common_idx) == 0:
                print(f"[INFO] No shared cells between plate {plate} and {tissue}")
                continue

            # Insert celltypes for those cells
            adata.obs.loc[common_idx, "celltype"] = filt_obs.loc[common_idx, "celltype"]

        # Store final annotated plate
        adata_raw_plate_with_celltype_dict[plate] = adata

    return adata_raw_plate_with_celltype_dict

In [ ]:
adata_raw_dict = load_8cubed_raw_data(adata_raw_parent_dir, debug=debug)
adata_filtered_tissue_dict = load_8cubed_filtered_data(adata_filtered_dir, debug=debug)

## Knee plot - use this output to estimate umi_cutoff

In [ ]:
expected_cells = {plate: 0 for plate in adata_raw_dict.keys()}
for adata in adata_filtered_tissue_dict.values():
    for plate in adata.obs["plate"].unique():
        if plate in expected_cells:
            # sum value counts from adata
            expected_cells[plate] += int(adata.obs.loc[adata.obs["plate"] == plate, "n_counts"].value_counts().sum())
expected_cells

In [ ]:
if not debug:
    umi_cutoffs = {}
    for plate, adata_raw in adata_raw_dict.items():
        umi_cutoff = cs_utils.knee_plot(adata_raw, expected_cells=expected_cells[plate], title=f"Knee Plot - {plate}", out_path=os.path.join(out_dir, f"knee_plot_{plate}.png"))
        umi_cutoffs[plate] = umi_cutoff

In [ ]:
#!!! optionally update umi_cutoffs from knee plot - required for None values
# umi_cutoffs = {
#     "igvf_003": None,
#     "igvf_004": None,
#     "igvf_005": None,
#     "igvf_007": None,
#     "igvf_008b": None,
#     "igvf_009": None,
#     "igvf_010": None,
#     "igvf_011": None,
# }

In [ ]:
adata_raw_dict = merge_filtered_celltype_into_raw(adata_raw_dict, adata_filtered_tissue_dict)

In [ ]:
if not debug:
    adata_raw_filtered_dict = {}
    for plate, adata_raw in adata_raw_dict.items():
        adata_raw = cs_utils.infer_empty_droplets(adata_raw, method="threshold", umi_cutoff=umi_cutoffs[plate], verbose=verbose)  # adds adata.obs["is_empty"]
        adata_raw.var['empty_counts'] = np.array(adata_raw.X[adata_raw.obs['is_empty'].values, :].sum(axis=0)).flatten()
        adata_raw_dict[plate] = adata_raw
        adata_raw_filtered = adata_raw[~adata_raw.obs["is_empty"]].copy()
        adata_raw_filtered_dict[plate] = adata_raw_filtered

## cellsweep

In [ ]:
if not debug:
    adata_cellsweep_dict = {}
    for plate, adata_raw in adata_raw_dict.items():
        print(f"Processing Cellsweep for plate {plate}...")
        adata_path_cellsweep = os.path.join(data_dir, plate, "cellsweep.h5ad")
        cellsweep_log_path = os.path.join(data_dir, plate, "cellsweep.log")
        if not os.path.exists(adata_path_cellsweep) or overwrite:
            adata_cellsweep = denoise_count_matrix(adata_raw, adata_out=adata_path_cellsweep, beta=cellsweep_beta, freeze_ambient_profile=True, init_alpha=cellsweep_init_alpha, max_iter=cellsweep_max_iter, empty_droplet_method="threshold", expected_cells=expected_cells[plate], threads=threads, verbose=verbose, log_file=cellsweep_log_path)
        else:
            adata_cellsweep = ad.read_h5ad(adata_path_cellsweep)
        adata_cellsweep = adata_cellsweep[~adata_cellsweep.obs["is_empty"]].copy()
        adata_cellsweep.var_names_make_unique()
        adata_cellsweep_dict[plate] = adata_cellsweep

# Analysis

In [ ]:
if debug:
    adata_raw_filtered_dict = adata_raw_dict.copy()
    adata_cellsweep_dict = adata_raw_dict.copy()

In [ ]:
for plate in adata_raw_dict.keys():
    print(f"Plate: {plate}")
    print(f"Adata raw (filtered cells): {adata_raw_filtered_dict[plate]}")
    print(f"Adata cellsweep: {adata_cellsweep_dict[plate]}\n\n")

## Save Anndatas

In [ ]:
for plate, adata in adata_raw_filtered_dict.items():
    adata_path = os.path.join(data_dir, plate, "raw_counts_removed_empty_barcodes.h5ad")
    os.makedirs(os.path.dirname(adata_path), exist_ok=True)
    adata.write_h5ad(adata_path)

# for plate, adata in adata_cellsweep_dict.items():
#     adata_path = os.path.join(data_dir, plate, "cellsweep.h5ad")
#     os.makedirs(os.path.dirname(adata_path), exist_ok=True)
#     adata.write_h5ad(adata_path)

## 8 Cubed Analysis

In [ ]:
# import os
# import anndata as ad
# import cellsweep.utils as cs_utils
# cellsweep_dir = os.path.dirname(os.path.abspath(""))
# data_dir = os.path.join(cellsweep_dir, "notebooks", "data", "8cubed")
# plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]
# adata_raw_filtered_dict, adata_cellsweep_dict = {}, {}
# for plate in plates:
#     adata_raw_filtered_dict[plate] = ad.read_h5ad(os.path.join(data_dir, plate, "raw_counts_removed_empty_barcodes.h5ad"))
#     adata_cellsweep_dict[plate] = ad.read_h5ad(os.path.join(data_dir, plate, "cellsweep.h5ad"))
# eight_cubed_markers_path = os.path.join(data_dir, "8_cube_marker_genes.csv")
# out_dir = os.path.join(cellsweep_dir, "notebooks", "output", "8cubed")

# import importlib
# import cellsweep.utils.visualization_utils as visualization_utils
# importlib.reload(visualization_utils)
# importlib.reload(cs_utils)

dict_of_adata_dicts = {
    "raw": adata_raw_filtered_dict,
    "cellsweep": adata_cellsweep_dict,
}
cs_utils.make_8cubed_plots(dict_of_adata_dicts, eight_cubed_markers_path, out_dir=out_dir)